In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import glob
import numpy as np
import pandas as pd

from PIL import Image
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder

Mounted at /content/drive


In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/ML_lab"
DATASET_DIR = os.path.join(DRIVE_DIR, "BreaKHis_v1")

CSV_FILES = glob.glob(os.path.join(DRIVE_DIR, "*.csv"))

CSV_PATH = CSV_FILES[0]

SAMPLES_PER_CLASS = 100

IMAGE_SIZE = (16, 16)

K = 3

DISTANCE_METRIC = "euclidean"

SORT_ALGORITHM = "insertion"

RANDOM_STATE = 42

In [ ]:
print(CSV_FILES)
print(DATASET_DIR)

In [ ]:
#A1

def resolve_image_path(filename):
    filename = str(filename).replace("\\", "/")

    path1 = os.path.join(DRIVE_DIR, filename)

    if os.path.exists(path1):
        return path1

    path2 = os.path.join(DATASET_DIR, filename)

    if os.path.exists(path2):
        return path2

    if filename.startswith("BreaKHis_v1/"):
        filename = filename[len("BreaKHis_v1/"):]

        path3 = os.path.join(DATASET_DIR, filename)

        if os.path.exists(path3):
            return path3

    return None


def extract_class(filename):
    filename = str(filename).lower()

    if "/benign/" in filename:
        return "benign"

    if "/malignant/" in filename:
        return "malignant"

    return np.nan


def extract_image_features(image_path, image_size=(16, 16)):
    image = Image.open(image_path).convert("L")
    image = image.resize(image_size)

    image_array = np.asarray(image, dtype=float)

    features = image_array.flatten()

    return features


def prepare_image_dataset(data, samples_per_class=100):
    data = data.copy()

    data["class"] = data["filename"].apply(extract_class)

    data = data.dropna(subset=["class"])

    sampled_data = (
        data.groupby("class", group_keys=False)
        .apply(
            lambda group: group.sample(
                n=min(samples_per_class, len(group)),
                random_state=RANDOM_STATE
            )
        )
        .reset_index(drop=True)
    )

    feature_rows = []
    labels = []

    for _, row in sampled_data.iterrows():

        image_path = resolve_image_path(row["filename"])

        if image_path is None:
            continue

        try:
            features = extract_image_features(
                image_path,
                IMAGE_SIZE
            )

            feature_rows.append(features)
            labels.append(row["class"])

        except Exception:
            continue

    X = pd.DataFrame(feature_rows)
    y = pd.Series(labels, name="class")

    return X, y


def encode_data(data):
    encoded_data = data.copy()

    categorical_columns = encoded_data.select_dtypes(
        include=["object", "category"]
    ).columns

    for column in categorical_columns:

        categories = encoded_data[column].dropna().unique()

        mapping = {
            value: index
            for index, value in enumerate(categories)
        }

        encoded_data[column] = encoded_data[column].map(mapping)

    return encoded_data


def impute_missing_values(data, method="mean"):
    imputed_data = data.copy()

    for column in imputed_data.columns:

        if imputed_data[column].isnull().any():

            if method == "mean":
                value = imputed_data[column].mean()

            elif method == "median":
                value = imputed_data[column].median()

            elif method == "mode":
                value = imputed_data[column].mode()[0]

            else:
                raise ValueError(
                    "Method must be mean, median, or mode."
                )

            imputed_data[column] = (
                imputed_data[column].fillna(value)
            )

    return imputed_data


def calculate_distance(point1, point2, metric="euclidean"):
    point1 = np.asarray(point1, dtype=float)
    point2 = np.asarray(point2, dtype=float)

    if metric == "euclidean":

        return np.sqrt(
            np.sum((point1 - point2) ** 2)
        )

    elif metric == "manhattan":

        return np.sum(
            np.abs(point1 - point2)
        )

    else:

        raise ValueError(
            "Metric must be euclidean or manhattan."
        )


def bubble_sort(items):
    result = items.copy()

    n = len(result)

    for i in range(n):

        for j in range(0, n - i - 1):

            key1 = (result[j][0], result[j][2])
            key2 = (result[j + 1][0], result[j + 1][2])

            if key1 > key2:

                result[j], result[j + 1] = (
                    result[j + 1],
                    result[j]
                )

    return result


def selection_sort(items):
    result = items.copy()

    n = len(result)

    for i in range(n):

        minimum_index = i

        for j in range(i + 1, n):

            key1 = (
                result[j][0],
                result[j][2]
            )

            key2 = (
                result[minimum_index][0],
                result[minimum_index][2]
            )

            if key1 < key2:
                minimum_index = j

        result[i], result[minimum_index] = (
            result[minimum_index],
            result[i]
        )

    return result


def insertion_sort(items):
    result = items.copy()

    for i in range(1, len(result)):

        current = result[i]

        j = i - 1

        while j >= 0:

            previous_key = (
                result[j][0],
                result[j][2]
            )

            current_key = (
                current[0],
                current[2]
            )

            if previous_key <= current_key:
                break

            result[j + 1] = result[j]

            j -= 1

        result[j + 1] = current

    return result


def sort_distances(items, algorithm="bubble"):

    if algorithm == "bubble":
        return bubble_sort(items)

    elif algorithm == "selection":
        return selection_sort(items)

    elif algorithm == "insertion":
        return insertion_sort(items)

    else:
        raise ValueError(
            "Use bubble, selection, or insertion."
        )


def find_neighbors(sorted_distances, k):

    if k <= 0:
        raise ValueError("k must be greater than zero.")

    return sorted_distances[:k]


def assign_class(neighbors):

    class_counts = Counter()

    for distance, class_label, index in neighbors:
        class_counts[class_label] += 1

    maximum_votes = max(class_counts.values())

    tied_classes = [
        label
        for label, count in class_counts.items()
        if count == maximum_votes
    ]

    if len(tied_classes) == 1:
        return tied_classes[0]

    for distance, class_label, index in neighbors:

        if class_label in tied_classes:
            return class_label


def custom_knn_predict(X_train, y_train, X_test, k=3,
                       metric="euclidean",
                       algorithm="insertion"):

    predictions = []

    for test_point in X_test:

        distance_list = []

        for index in range(len(X_train)):

            distance = calculate_distance(
                test_point,
                X_train[index],
                metric
            )

            distance_list.append(
                (
                    distance,
                    y_train[index],
                    index
                )
            )

        sorted_distances = sort_distances(
            distance_list,
            algorithm
        )

        neighbors = find_neighbors(
            sorted_distances,
            k
        )

        prediction = assign_class(neighbors)

        predictions.append(prediction)

    return np.array(predictions)

In [ ]:
#A2

def weighted_class_assignment(neighbors):

    class_weights = {}

    for distance, class_label, index in neighbors:

        weight = 1 / (distance + 1e-10)

        class_weights[class_label] = (
            class_weights.get(class_label, 0) + weight
        )

    maximum_weight = max(class_weights.values())

    tied_classes = [
        label
        for label, weight in class_weights.items()
        if weight == maximum_weight
    ]

    if len(tied_classes) == 1:
        return tied_classes[0]

    for distance, class_label, index in neighbors:

        if class_label in tied_classes:
            return class_label


def weighted_knn_predict(X_train, y_train, X_test,
                         k=3,
                         metric="euclidean",
                         algorithm="insertion"):

    predictions = []

    for test_point in X_test:

        distance_list = []

        for index in range(len(X_train)):

            distance = calculate_distance(
                test_point,
                X_train[index],
                metric
            )

            distance_list.append(
                (
                    distance,
                    y_train[index],
                    index
                )
            )

        sorted_distances = sort_distances(
            distance_list,
            algorithm
        )

        neighbors = find_neighbors(
            sorted_distances,
            k
        )

        prediction = weighted_class_assignment(
            neighbors
        )

        predictions.append(prediction)

    return np.array(predictions)

In [ ]:
#A3

def split_dataset(X, y, test_size=0.2,
                  random_state=42):

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
        stratify=y
    )

    return (
        X_train,
        X_test,
        y_train,
        y_test
    )

In [ ]:
#A4

def train_sklearn_knn(X_train, y_train, k=3):

    model = KNeighborsClassifier(
        n_neighbors=k
    )

    model.fit(
        X_train,
        y_train
    )

    return model

In [ ]:
#A5

def calculate_accuracy(y_actual, y_predicted):

    return accuracy_score(
        y_actual,
        y_predicted
    )

In [ ]:
#A6

def get_predictions(model, X_test):

    predictions = model.predict(
        X_test
    )

    return predictions

def prediction_table(y_actual, y_predicted):

    result = pd.DataFrame({
        "Actual": y_actual,
        "Predicted": y_predicted
    })

    result["Correct"] = (
        result["Actual"] == result["Predicted"]
    )

    return result

In [ ]:
#A7

class CustomKNN:

    def __init__(self, k=3,
                 metric="euclidean",
                 algorithm="insertion"):

        self.k = k
        self.metric = metric
        self.algorithm = algorithm

        self.X_train = None
        self.y_train = None


    def fit(self, X_train, y_train):

        self.X_train = np.asarray(
            X_train,
            dtype=float
        )

        self.y_train = np.asarray(
            y_train
        )

        return self


    def predict(self, X_test):

        X_test = np.asarray(
            X_test,
            dtype=float
        )

        return custom_knn_predict(
            self.X_train,
            self.y_train,
            X_test,
            self.k,
            self.metric,
            self.algorithm
        )


    def score(self, X_test, y_test):

        predictions = self.predict(
            X_test
        )

        return accuracy_score(
            y_test,
            predictions
        )